# Perfect foresight material stockpiling

This minimal example tests whether a material can be stored across investment periods with `stateOfChargeBoundary="interInvestmentPeriod"`. The intra-year storage trajectory is only a modelling helper; the relevant values are the SOC values at investment-period boundaries.

## 1. Import packages

In [1]:
import fine as fn
import pandas as pd
import pyomo.environ as pyomo

## 2. Create the energy system model

In [2]:
loc = "TestRegion"
startYear = 2020
numberOfInvestmentPeriods = 2
interval = 1

esM = fn.EnergySystemModel(
    locations={loc},
    commodities={"electricity"},
    commodityUnitsDict={"electricity": "MW_el"},
    materials={"steel"},
    materialUnitsDict={"steel": "t"},
    numberOfTimeSteps=1,
    hoursPerTimeStep=1,
    numberOfInvestmentPeriods=numberOfInvestmentPeriods,
    investmentPeriodInterval=interval,
    startYear=startYear,
    costUnit="1 Euro",
    lengthUnit="km",
    verboseLogLevel=0,
)

## 3. Add a component which needs steel in IP 2021

The dummy plant is commissioned only in the second investment period. With a steel intensity of 1 t/MW, commissioning 10 MW creates a steel demand of 10 t in IP 2021.

In [3]:
zero = pd.Series({loc: 0.0})
ten = pd.Series({loc: 10.0})

esM.add(
    fn.Source(
        esM=esM,
        name="Steel intensive plant",
        commodity="electricity",
        hasCapacityVariable=True,
        commissioningFix={2020: zero, 2021: ten},
        investPerCapacity=0,
        opexPerCapacity=0,
        economicLifetime=10,
        technicalLifetime=10,
        materialIntensity={
            2020: {"steel": pd.Series({loc: 1.0})},
            2021: {"steel": pd.Series({loc: 1.0})},
        },
    )
)

## 4. Add steel supply and steel demand sink

Steel is produced only in IP 2020. The material demand sink is the artificial sink used by FINE's material demand constraint.

In [4]:
steel_supply = {
    2020: pd.DataFrame([[10.0]], index=[0], columns=[loc]),
    2021: pd.DataFrame([[0.0]], index=[0], columns=[loc]),
}

esM.add(
    fn.Source(
        esM=esM,
        name="Steel supply",
        commodity="steel",
        hasCapacityVariable=False,
        operationRateFix=steel_supply,
    )
)

esM.add(
    fn.Sink(
        esM=esM,
        name="Steel demand",
        commodity="steel",
        hasCapacityVariable=False,
        material=True,
    )
)

## 5. Add the material stockpile

The storage is a `MaterialStorage`. Its charge and discharge operations only link the stockpile to the material commodity balance; the relevant stock trajectory is represented by IP-level start and end stock variables.

In [5]:
# esM.add(
#     fn.Storage(
#         esM=esM,
#         name="Steel stockpile",
#         commodity="steel",
#         hasCapacityVariable=True,
#         investPerCapacity=1,
#         opexPerCapacity=0,
#         economicLifetime=10,
#         technicalLifetime=10,
#         chargeEfficiency=1,
#         dischargeEfficiency=1,
#         selfDischarge=0,
#         chargeRate=1,
#         dischargeRate=1,
#         stateOfChargeBoundary="interInvestmentPeriod",
#     )
# )

In [6]:
esM.add(
    fn.MaterialStorage(
        esM=esM,
        name="Steel stockpile",
        commodity="steel",
        hasCapacityVariable=True,
        investPerCapacity=1,
        opexPerCapacity=0,
        economicLifetime=10,
        technicalLifetime=10,
        chargeEfficiency=1,
        dischargeEfficiency=1,
        selfDischarge=0,
        stateOfChargeBoundary="interInvestmentPeriod",
    )
)

## 6. Optimize

In [7]:
esM.optimize(timeSeriesAggregation=False, solver="glpk")

Declaring sets, variables and constraints for SourceSinkModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(0.0124 sec)

Declaring sets, variables and constraints for MaterialStorageModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(0.0037 sec)

Declaring shared potential constraint...
		(0.0010 sec)

Declaring linked component quantity constraint...
		(0.0000 sec)

Declaring commodity balances...
		(0.0007 sec)

Declaring material demand constraints...
LHS_demand op_srcSnk[TestRegion,Steel demand,0,0,0]
RHS_demand commis_srcSnk[TestRegion,Steel intensive plant,0]
LHS_demand op_srcSnk[TestRegion,Steel demand,1,0,0]
RHS_demand commis_srcSnk[TestRegion,Steel intensive plant,1]
		(0.0006 sec)

Declaring material recovery constraints...
		(0.0002 sec)

		(0.0000 sec)

Declaring objective function...
		(0.0266 sec)

GLPSOL--GLPK LP/MIP Solver 5.0
Parameter(s) specified in the command line:
 --write C:\Users\JF3E6~1.MUT\AppData\Local

## 7. Check the relevant results

In [8]:
def src_sink_sum(component_name, ip):
    return sum(
        pyomo.value(esM.pyM.op_srcSnk[loc, component_name, ip, p, t])
        * esM.periodOccurrences[ip][p]
        for p, t in esM.pyM.intraYearTimeSet
        if (loc, component_name, ip, p, t) in esM.pyM.op_srcSnk
    )


def storage_sum(var_name, component_name, ip):
    var = getattr(esM.pyM, var_name)
    return sum(
        pyomo.value(var[loc, component_name, ip, p, t])
        * esM.periodOccurrences[ip][p]
        for p, t in esM.pyM.intraYearTimeSet
        if (loc, component_name, ip, p, t) in var
    )


stock_start = esM.pyM.stateOfChargeStart_matStor
stock_end = esM.pyM.stateOfChargeEnd_matStor
cap = esM.pyM.cap_matStor
storage_name = "Steel stockpile"
rows = []

for ip_year in esM.investmentPeriodNames:
    ip = esM.investmentPeriodNames.index(ip_year)
    rows.append(
        {
            "IP": ip_year,
            "steel_supply": src_sink_sum("Steel supply", ip),
            "steel_demand": src_sink_sum("Steel demand", ip),
            "storage_capacity": pyomo.value(cap[loc, storage_name, ip]),
            "storage_charge": storage_sum("chargeOp_matStor", storage_name, ip),
            "storage_discharge": storage_sum("dischargeOp_matStor", storage_name, ip),
            "stock_start": pyomo.value(stock_start[loc, storage_name, ip]),
            "stock_end": pyomo.value(stock_end[loc, storage_name, ip]),
        }
    )

results = pd.DataFrame(rows)
results

,IP,steel_supply,steel_demand,storage_capacity,storage_charge,storage_discharge,stock_start,stock_end
0,2020,10.0,0.0,10.0,10.0,0.0,0.0,10.0
1,2021,0.0,10.0,10.0,0.0,10.0,10.0,0.0


Expected pattern: steel is supplied in IP 2020, consumed in IP 2021, and the stockpile links both IPs.

In [9]:
tol = 1e-6

assert abs(results.loc[0, "steel_supply"] - 10) < tol
assert abs(results.loc[0, "steel_demand"] - 0) < tol
assert abs(results.loc[1, "steel_supply"] - 0) < tol
assert abs(results.loc[1, "steel_demand"] - 10) < tol

assert all(results["storage_capacity"] >= 10 - tol)

assert abs(results.loc[0, "stock_end"] - results.loc[1, "stock_start"]) < tol
assert abs(results.loc[1, "stock_end"] - results.loc[0, "stock_start"]) < tol

assert abs(
    results.loc[0, "stock_end"]
    - results.loc[0, "stock_start"]
    - results.loc[0, "steel_supply"]
) < tol
assert abs(
    results.loc[1, "stock_end"]
    - results.loc[1, "stock_start"]
    + results.loc[1, "steel_demand"]
) < tol

print("Material stockpiling check passed.")

Material stockpiling check passed.


## 8. Optional summaries

In [10]:
for year in esM.investmentPeriodNames:
    print(f"\nResults of SourceSinkModel for year {year}")
    print(esM.getOptimizationSummary("SourceSinkModel", outputLevel=2, ip=year))
    print(f"\nResults of MaterialStorageModel for year {year}")
    print(esM.getOptimizationSummary("MaterialStorageModel", outputLevel=2, ip=year))


Results of SourceSinkModel for year 2020
                               TestRegion
Component    Property  Unit              
Steel supply operation [t*h/a]    87600.0
                       [t*h]         10.0

Results of MaterialStorageModel for year 2020
                                            TestRegion
Component       Property         Unit                 
Steel stockpile NPVcontribution  [1 Euro]     1.490295
                TAC              [1 Euro/a]   1.490295
                capacity         [t*h]            10.0
                capexCap         [1 Euro/a]   1.490295
                commissioning    [t*h]            10.0
                invest           [1 Euro]         10.0
                operationCharge  [t*h]            10.0
                stateOfChargeEnd [t*h]            10.0

Results of SourceSinkModel for year 2021
                                            TestRegion
Component             Property      Unit              
Steel demand          operation     [t*h/